# 10 Hedge Ratio — M_n vs T12 (OLS Minimum Variance)

For each instrument M1-M8 (and T18/T24 as reference), compute the OLS hedge ratio against T12:

```
ΔM_n = α + H* × ΔT12 + ε
H* = Cov(ΔM_n, ΔT12) / Var(ΔT12) = ρ × σ(M_n) / σ(T12)
```

`H*` is the T12 notional that minimizes residual variance of the hedged portfolio.
Residual (unhedgeable) variance = Var(ΔM_n) × (1 − ρ²)

1. **Full-period OLS results** — H*, ρ, R², residual std per instrument
2. **Hedge effectiveness** — how much risk T12 actually removes
3. **Scatter plots** — M_n vs T12 daily changes with regression line
4. **Rolling hedge ratio** — time-stability of H* (252-day window)
5. **Pre-MPM vs off-MPM** — does hedge ratio change near meetings?

In [ ]:
import sys, warnings
sys.path.insert(0, '..')
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from scipy import stats

from src.processing import load_and_clean_data
from src.features import generate_features
from src.pooling import pool_boj_data

EXCEL_PATH  = '../data/BOJ_data.xlsx'
MEETING_CSV = '../data/BOJ_meeting_history.csv'

df_raw    = load_and_clean_data(EXCEL_PATH, MEETING_CSV)
df_feat   = generate_features(df_raw)

print(f'Shape: {df_feat.shape}')
print(f'Date range: {df_feat["Date"].min().date()} to {df_feat["Date"].max().date()}')

In [ ]:
# Compute daily spread changes for all instruments
# Use spread (= rate - policy rate) so that policy rate level changes are removed
spread_cols = ([f'M{i}_spread' for i in range(1, 9)] +
               [c for c in ['T12_spread', 'T18_spread', 'T24_spread'] if c in df_feat.columns])

delta = (
    df_feat.set_index('Date')[spread_cols + ['Is_Meeting_Day', 'Days_to_MPM']]
    .copy()
)
delta[spread_cols] = delta[spread_cols].diff()  # daily change
delta = delta.dropna(subset=spread_cols).copy()

# Instrument labels for display
BOJ_INSTRUMENTS = [f'M{i}' for i in range(1, 9)]
TENOR_INSTRUMENTS = [c.replace('_spread', '') for c in spread_cols if c.startswith('T')]
ALL_INSTRUMENTS  = BOJ_INSTRUMENTS + TENOR_INSTRUMENTS
col_map = {f'{inst}_spread': inst for inst in ALL_INSTRUMENTS}
delta = delta.rename(columns=col_map)

print(f'Daily changes computed: {len(delta)} rows')
print(f'Instruments: {ALL_INSTRUMENTS}')
print()
print('Daily change std (bps):')
print((delta[ALL_INSTRUMENTS].std() * 100).round(4).rename('std(bps)').to_frame().T)

## 1. Full-Period OLS Results

For each M_n (and T18/T24 as reference), regress daily spread change on ΔT12.
Using the full available history for stable coefficient estimates.

In [ ]:
def ols_hedge(y_series, x_series):
    """OLS: y = α + H*x + ε.  Returns dict of key stats."""
    df = pd.DataFrame({'y': y_series, 'x': x_series}).dropna()
    if len(df) < 30:
        return None
    slope, intercept, r, p, se = stats.linregress(df['x'], df['y'])
    sigma_y   = df['y'].std()
    sigma_x   = df['x'].std()
    resid_std = sigma_y * np.sqrt(1 - r**2)   # unhedgeable residual std
    vol_ratio = sigma_y / sigma_x              # naive vol-ratio hedge (for comparison)
    return {
        'H* (OLS beta)'      : round(slope, 4),
        'Vol Ratio (ref)'    : round(vol_ratio, 4),
        'rho'                : round(r, 4),
        'R2 (%)'             : round(r**2 * 100, 1),
        'sigma_Mn (bps)'     : round(sigma_y * 100, 4),
        'Residual std (bps)' : round(resid_std * 100, 4),
        'Hedge Eff. (%)'     : round(r**2 * 100, 1),
        'N'                  : len(df),
    }

results = {}
for inst in [i for i in ALL_INSTRUMENTS if i != 'T12']:
    res = ols_hedge(delta[inst], delta['T12'])
    if res:
        results[inst] = res

df_ols = pd.DataFrame(results).T
print('=== OLS Hedge Ratio: each instrument vs T12 ===')
df_ols

## 2. Hedge Effectiveness Visualization

In [ ]:
boj_instruments = [i for i in BOJ_INSTRUMENTS if i in df_ols.index]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('OLS Hedge Ratio vs T12 — Full Period', fontsize=13)

# (1) Hedge ratio H*
ax = axes[0]
beta = df_ols.loc[boj_instruments, 'H* (OLS beta)'].astype(float)
vol_r = df_ols.loc[boj_instruments, 'Vol Ratio (ref)'].astype(float)
x = np.arange(len(boj_instruments))
ax.bar(x - 0.2, beta.values,  0.38, label='OLS beta (H*)',   color='steelblue', edgecolor='black', linewidth=0.5)
ax.bar(x + 0.2, vol_r.values, 0.38, label='Vol ratio (ref)', color='lightgray',  edgecolor='black', linewidth=0.5)
ax.axhline(0, color='black', linewidth=0.6)
ax.set_xticks(x)
ax.set_xticklabels(boj_instruments)
ax.set_title('Hedge Ratio H* (T12 notional per M_n unit)')
ax.set_ylabel('Hedge Ratio')
ax.legend(fontsize=9)

# (2) Correlation ρ
ax = axes[1]
rho = df_ols.loc[boj_instruments, 'rho'].astype(float)
colors = ['steelblue' if v >= 0 else 'salmon' for v in rho]
ax.bar(boj_instruments, rho.values, color=colors, edgecolor='black', linewidth=0.5)
for i, v in enumerate(rho.values):
    ax.text(i, v + 0.01 * (1 if v >= 0 else -1), f'{v:.3f}', ha='center', fontsize=9)
ax.axhline(0, color='black', linewidth=0.6)
ax.set_title('Correlation (rho) with T12')
ax.set_ylabel('rho')
ax.set_ylim(-0.1, 1.05)

# (3) Residual std (unhedgeable risk)
ax = axes[2]
sigma_mn  = df_ols.loc[boj_instruments, 'sigma_Mn (bps)'].astype(float)
resid_std = df_ols.loc[boj_instruments, 'Residual std (bps)'].astype(float)
ax.bar(x - 0.2, sigma_mn.values,  0.38, label='sigma(M_n) unhedged', color='salmon',    edgecolor='black', linewidth=0.5)
ax.bar(x + 0.2, resid_std.values, 0.38, label='Residual std hedged', color='steelblue', edgecolor='black', linewidth=0.5)
ax.set_xticks(x)
ax.set_xticklabels(boj_instruments)
ax.set_title('Risk Before vs After Hedge (bps/day)')
ax.set_ylabel('Std (bps/day)')
ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

## 3. Scatter Plots — M_n vs T12 Daily Changes

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(20, 9))
axes = axes.flatten()
fig.suptitle('Daily Spread Change: M_n vs T12  (regression line = OLS hedge)', fontsize=13)

for i, inst in enumerate(boj_instruments):
    ax = axes[i]
    df_plot = pd.DataFrame({'x': delta['T12'] * 100, 'y': delta[inst] * 100}).dropna()
    beta_val = float(df_ols.loc[inst, 'H* (OLS beta)'])
    rho_val  = float(df_ols.loc[inst, 'rho'])
    r2_val   = float(df_ols.loc[inst, 'R2 (%)'])

    ax.scatter(df_plot['x'], df_plot['y'], alpha=0.25, s=8, color='steelblue')
    xlim = ax.get_xlim()
    xline = np.linspace(df_plot['x'].min(), df_plot['x'].max(), 100)
    ax.plot(xline, beta_val * xline, color='red', linewidth=1.5,
            label=f'H*={beta_val:.3f}  rho={rho_val:.3f}  R2={r2_val:.1f}%')
    ax.axhline(0, color='black', linewidth=0.4)
    ax.axvline(0, color='black', linewidth=0.4)
    ax.set_title(inst, fontweight='bold')
    ax.set_xlabel('DeltaT12 (bps/day)')
    ax.set_ylabel(f'Delta{inst} (bps/day)')
    ax.legend(fontsize=7.5, loc='upper left')

plt.tight_layout()
plt.show()

## 4. Rolling Hedge Ratio (252-day window)

Checks whether H* is stable over time or changes with market regime.

In [ ]:
WINDOW = 252

def rolling_beta(y, x, window):
    """Rolling OLS beta using expanding arrays."""
    betas = [np.nan] * (window - 1)
    for end in range(window, len(y) + 1):
        yy = y[end - window:end]
        xx = x[end - window:end]
        mask = ~(np.isnan(yy) | np.isnan(xx))
        if mask.sum() < 30:
            betas.append(np.nan)
            continue
        slope, *_ = stats.linregress(xx[mask], yy[mask])
        betas.append(slope)
    return np.array(betas)

fig, axes = plt.subplots(2, 4, figsize=(20, 9))
axes = axes.flatten()
fig.suptitle(f'Rolling Hedge Ratio H* ({WINDOW}-day window) — M_n vs T12', fontsize=13)

dates_arr = delta.index.values
x_arr = delta['T12'].values

for i, inst in enumerate(boj_instruments):
    ax = axes[i]
    y_arr  = delta[inst].values
    betas  = rolling_beta(y_arr, x_arr, WINDOW)
    full_beta = float(df_ols.loc[inst, 'H* (OLS beta)'])

    ax.plot(dates_arr, betas, color='steelblue', linewidth=1.0, label='Rolling H*')
    ax.axhline(full_beta, color='red', linestyle='--', linewidth=1.2,
               label=f'Full-period H*={full_beta:.3f}')
    ax.axhline(0, color='black', linewidth=0.5)
    ax.set_title(inst, fontweight='bold')
    ax.set_ylabel('H*')
    ax.legend(fontsize=8)
    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right', fontsize=7)

plt.tight_layout()
plt.show()

## 5. Pre-MPM vs Off-MPM Hedge Ratio

Near MPM (Days_to_MPM <= 5), M1-M3 tend to move independently as the market prices in the specific meeting outcome.
Does the hedge ratio change significantly near meetings?

In [ ]:
pre_mpm_mask  = delta['Days_to_MPM'] <= 5
off_mpm_mask  = delta['Days_to_MPM'] >  5

rows = []
for inst in boj_instruments:
    for label, mask in [('Pre-MPM (<=5d)', pre_mpm_mask), ('Off-MPM (>5d)', off_mpm_mask)]:
        sub = delta[mask]
        res = ols_hedge(sub[inst], sub['T12'])
        if res:
            rows.append({'Instrument': inst, 'Period': label,
                         'H*': res['H* (OLS beta)'],
                         'rho': res['rho'],
                         'R2 (%)': res['R2 (%)'],
                         'Residual std (bps)': res['Residual std (bps)'],
                         'N': res['N']})

df_regime = pd.DataFrame(rows)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Hedge Ratio H* and Correlation — Pre-MPM vs Off-MPM', fontsize=13)

for ax, metric, ylabel in [
        (axes[0], 'H*',   'Hedge Ratio H*'),
        (axes[1], 'rho',  'Correlation rho')]:
    pre  = df_regime[df_regime['Period'] == 'Pre-MPM (<=5d)'].set_index('Instrument')[metric]
    off  = df_regime[df_regime['Period'] == 'Off-MPM (>5d)'].set_index('Instrument')[metric]
    x    = np.arange(len(boj_instruments))
    pre_vals = pre.reindex(boj_instruments).values.astype(float)
    off_vals = off.reindex(boj_instruments).values.astype(float)
    ax.bar(x - 0.2, pre_vals, 0.38, label='Pre-MPM (<=5d)', color='coral',     edgecolor='black', linewidth=0.5)
    ax.bar(x + 0.2, off_vals, 0.38, label='Off-MPM (>5d)',  color='steelblue', edgecolor='black', linewidth=0.5)
    ax.axhline(0, color='black', linewidth=0.5)
    ax.set_xticks(x)
    ax.set_xticklabels(boj_instruments)
    ax.set_title(ylabel)
    ax.set_ylabel(ylabel)
    ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

print('\n=== Detailed table ===')
print(df_regime.pivot(index='Instrument', columns='Period',
                      values=['H*','rho','R2 (%)','Residual std (bps)']).round(3).to_string())

## 6. Summary — Practical Hedge Table

For 1 unit of M_n, how much T12 to trade as hedge?

## 7. DV01-Adjusted Notional Hedge Ratio

OLS beta H* is the **rate-level** hedge: for 1 unit of M_n, short H* units of T12.
But M_n and T12 have different tenors → different DV01 → different PnL per 1bp rate move.

To make the hedge **PnL-neutral** (same $ gain/loss per 1bp), scale by the DV01 ratio:

```
Notional_T12 / Notional_Mn = H* × DV01(M_n) / DV01(T12)
                            ≈ H* × tenor_Mn_years / 1.0
```

Since M_n is the nth-forward OIS starting ~now and ending at the nth BOJ meeting:
- M1 tenor ≈ average Days_to_MPM  
- M_n tenor ≈ M1 tenor + (n−1) × avg meeting interval

**Interpretation**: For 1bp move in M_n with notional N, the T12 notional needed to offset the PnL is `N × (Notional hedge ratio)`.

In [ ]:
# ── DV01-adjusted notional hedge ────────────────────────────────────────────
# Step 1: estimate M_n tenors from meeting history
meetings_df = pd.read_csv(MEETING_CSV, parse_dates=['Date'])
meetings_df = meetings_df.sort_values('Date').reset_index(drop=True)

avg_interval_days = meetings_df['Date'].diff().dt.days.dropna().mean()
avg_days_to_m1    = df_feat['Days_to_MPM'].mean()   # average days until next BOJ meeting

# M_n tenor (days): M1 = avg_days_to_m1, M2 = M1 + interval, ...
tenor_days = {f'M{n}': avg_days_to_m1 + (n - 1) * avg_interval_days for n in range(1, 9)}
tenor_T12_days = 365.25   # 12m OIS

print(f'Average meeting interval : {avg_interval_days:.1f} days')
print(f'Average Days_to_MPM (M1) : {avg_days_to_m1:.1f} days')
print()
print('Estimated M_n tenors:')
for inst, t in tenor_days.items():
    print(f'  {inst}: {t:.0f} days ({t/365:.2f} yr)')

# ── Step 2: compute DV01 ratio and notional hedge ───────────────────────────
rows = []
for inst in boj_instruments:
    if inst not in df_ols.index:
        continue
    h_star     = float(df_ols.loc[inst, 'H* (OLS beta)'])
    rho        = float(df_ols.loc[inst, 'rho'])
    r2         = float(df_ols.loc[inst, 'R2 (%)'])
    sigma_mn   = float(df_ols.loc[inst, 'sigma_Mn (bps)'])
    resid_std  = float(df_ols.loc[inst, 'Residual std (bps)'])

    t_mn       = tenor_days[inst]
    dv01_ratio = t_mn / tenor_T12_days          # DV01(M_n) / DV01(T12)
    n_hedge    = h_star * dv01_ratio             # notional T12 per 1 unit M_n
    pnl_resid  = resid_std * dv01_ratio          # residual PnL std in T12-DV01 equiv (bps)

    rows.append({
        'Instrument'              : inst,
        'Tenor (days)'            : round(t_mn, 0),
        'DV01 ratio (Mn/T12)'     : round(dv01_ratio, 3),
        'H* (rate hedge)'         : round(h_star, 3),
        'Notional hedge (T12/Mn)' : round(n_hedge, 3),
        'rho'                     : round(rho, 3),
        'R2 (%)'                  : round(r2, 1),
        'Unhedged sigma (bps/d)'  : round(sigma_mn, 3),
        'Hedged resid (bps/d)'    : round(resid_std, 3),
        'Risk reduction (%)'      : round((1 - resid_std / sigma_mn) * 100, 1),
    })

df_notional = pd.DataFrame(rows).set_index('Instrument')

print()
print('=== DV01-Adjusted Notional Hedge Table ===')
print(df_notional.to_string())
print()
print('Interpretation of "Notional hedge (T12/Mn)":')
print('  For 1bp move in M_n, the PnL-equivalent T12 notional to short = Notional_Mn × (this value)')

# ── Step 3: bar chart ────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('DV01-Adjusted Notional Hedge: M_n vs T12', fontsize=13)

x = np.arange(len(boj_instruments))

# (1) H* vs Notional hedge side by side
ax = axes[0]
h_vals  = df_notional['H* (rate hedge)'].values.astype(float)
nh_vals = df_notional['Notional hedge (T12/Mn)'].values.astype(float)
ax.bar(x - 0.2, h_vals,  0.38, label='H* (rate, same face)',       color='lightgray',  edgecolor='black', linewidth=0.5)
ax.bar(x + 0.2, nh_vals, 0.38, label='Notional hedge (DV01-adj.)', color='steelblue',  edgecolor='black', linewidth=0.5)
ax.axhline(0, color='black', linewidth=0.5)
ax.set_xticks(x)
ax.set_xticklabels(boj_instruments)
ax.set_title('Rate hedge H* vs DV01-adjusted Notional')
ax.set_ylabel('Hedge ratio')
ax.legend(fontsize=8)

# (2) Correlation
ax = axes[1]
rho_vals = df_notional['rho'].values.astype(float)
colors   = ['steelblue' if v >= 0 else 'salmon' for v in rho_vals]
ax.bar(boj_instruments, rho_vals, color=colors, edgecolor='black', linewidth=0.5)
for i, v in enumerate(rho_vals):
    ax.text(i, v + 0.01, f'{v:.3f}', ha='center', fontsize=9)
ax.set_title('Correlation rho with T12')
ax.set_ylabel('rho')
ax.set_ylim(-0.1, 1.05)

# (3) Unhedged vs hedged residual (bps/day)
ax = axes[2]
u_vals = df_notional['Unhedged sigma (bps/d)'].values.astype(float)
h_vals2 = df_notional['Hedged resid (bps/d)'].values.astype(float)
ax.bar(x - 0.2, u_vals,  0.38, label='Unhedged sigma', color='salmon',    edgecolor='black', linewidth=0.5)
ax.bar(x + 0.2, h_vals2, 0.38, label='Hedged residual', color='steelblue', edgecolor='black', linewidth=0.5)
ax.set_xticks(x)
ax.set_xticklabels(boj_instruments)
ax.set_title('Daily Rate Risk Before/After Hedge (bps)')
ax.set_ylabel('Std (bps/day)')
ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# Build practical summary
summary_rows = []
for inst in boj_instruments:
    full = df_ols.loc[inst]
    pre  = df_regime[(df_regime['Instrument']==inst) & (df_regime['Period']=='Pre-MPM (<=5d)')].iloc[0]
    off  = df_regime[(df_regime['Instrument']==inst) & (df_regime['Period']=='Off-MPM (>5d)')].iloc[0]
    summary_rows.append({
        'Instrument'           : inst,
        'H* (full period)'     : float(full['H* (OLS beta)']),
        'H* (pre-MPM)'         : float(pre['H*']),
        'H* (off-MPM)'         : float(off['H*']),
        'rho (full)'           : float(full['rho']),
        'R2 % (full)'          : float(full['R2 (%)']),
        'sigma unhedged (bps)' : float(full['sigma_Mn (bps)']),
        'sigma hedged (bps)'   : float(full['Residual std (bps)']),
        'Risk reduction %'     : round((1 - float(full['Residual std (bps)'])/float(full['sigma_Mn (bps)'])) * 100, 1),
    })

df_summary = pd.DataFrame(summary_rows).set_index('Instrument')
print('=== Practical Hedge Table: M_n vs T12 ===')
print(df_summary.round(3).to_string())
print()
print('Interpretation:')
print('  H* (full period): for 1 unit long M_n, sell H* units of T12 to minimize variance')
print('  Risk reduction %: % of daily variance eliminated by the T12 hedge')